Τα HAPS notebooks δεν απαιτούν αλλαγές στον κώδικα, καθώς η λειτουργία τους βασίζεται σε δυναμική ανάγνωση των ενεργών portfolios από τον configuration πίνακα.

Για την προσθήκη του νέου χαρτοφυλακίου Astraea, αρκεί η εκτέλεση του σχετικού Databricks notebook, το οποίο εισάγει τις απαραίτητες εγγραφές στον πίνακα p-reportinputfiles.repoint.haps_repointportfolios.

In [0]:
from pyspark.sql import SparkSession
from datetime import datetime

spark = SparkSession.builder.getOrCreate()

env_prefix = "d"
portfolio_name = "Astraea"
business_date = datetime.now().strftime("%Y-%m-%d")
target_table = f"`{env_prefix}-reportinputfiles`.repoint.haps_repointportfolios"

# Κάθε HAPS notebook φιλτράρει τον haps_repointportfolios με WHERE A.Report = '<report_name>'.
# Χωρίς εγγραφή για κάθε Report, το notebook δεν θα "δει" το Astraea.

reports = [
    "HAPS_AssignmentMapping",
    "HAPS_CollectionsReport",
    "HAPS_FLDTApplications",
    "HAPS_FLDTModifications",
    "HAPS_L3869Modifications",
    "HAPS_L3869Perimeter",
    "HAPS_ModificationsMonitoring",
    "HAPS_ModificationsPerformance",
    "HAPS_PayerNonPayerWaterfall",
    "HAPS_PaymentsMtD",
    "HAPS_PLNPLWaterfall",
    "HAPS_Redefaults",
]


In [0]:
#domi pinaka
print(f"Schema του {target_table}:")
spark.sql(f"DESCRIBE {target_table}").show(truncate=False)

#elegxos an iparxei to Astrea

existing = spark.sql(f"""
    SELECT Report, PortfolioName, IsActive, BusinessDate, IngestionTimeStamp
    FROM {target_table}
    WHERE PortfolioName = '{portfolio_name}'
""")
existing_count = existing.count()
print(f"Υπάρχουσες εγγραφές για '{portfolio_name}': {existing_count}")
if existing_count > 0:
    existing.show(truncate=False)

#an iparxei fere mou ena deigma limit 5

sample = spark.sql(f"""
    SELECT *
    FROM {target_table}
    WHERE IsActive = 1
    LIMIT 5
""")
sample.show(truncate=False)
print(f"Columns: {sample.columns}")

In [0]:
#an iparxei fere mou ena deigma limit 5

sample = spark.sql(f"""
    SELECT *
    FROM {target_table}
    WHERE IsActive = 1
    LIMIT 5
""")
sample.show(truncate=False)
print(f"Columns: {sample.columns}")

In [0]:
# INSERT

for report in reports:
    check = spark.sql(f"""
        SELECT COUNT(*) AS cnt
        FROM {target_table}
        WHERE PortfolioName = '{portfolio_name}'
        AND Report = '{report}'
        AND IsActive = 1
    """).collect()[0]["cnt"]

    if check > 0:
        print(f"  [SKIP] {report} — υπάρχει ήδη")
        continue

    spark.sql(f"""
        INSERT INTO {target_table}
        (BusinessDate, IngestionTimeStamp, PortfolioName, Report, IsActive)
        VALUES
        ('{business_date}', current_timestamp(), '{portfolio_name}', '{report}', 1)
    """)
    print(f"  [OK] {report} — inserted")

# epalitheusi

verification = spark.sql(f"""
    SELECT Report, PortfolioName, IsActive, BusinessDate, IngestionTimeStamp
    FROM {target_table}
    WHERE PortfolioName = '{portfolio_name}'
    ORDER BY Report
""")
print(f"\nΕγγραφές για '{portfolio_name}': {verification.count()}")
verification.show(truncate=False)

# Σύγκριση με υπάρχοντα portfolios
print("\nΌλα τα ενεργά portfolios ανά report:")
spark.sql(f"""
    SELECT Report, 
           COLLECT_SET(PortfolioName) AS Portfolios,
           COUNT(DISTINCT PortfolioName) AS PortfolioCount
    FROM {target_table}
    WHERE IsActive = 1
    GROUP BY Report
    ORDER BY Report
""").show(truncate=False)
